# Notebook 02 — Business Decisions with Sample Paths

**Module 03: Sample Paths — The Correct Uncertainty Framework**

**Time:** ~15 minutes

## Learning Objectives

By the end of this notebook you will be able to:

1. Apply the Monte Carlo framework (simulate → apply → aggregate) to real business questions
2. Compute the 80th-percentile weekly total from sample paths
3. Compute the worst-case single day at a given service level
4. Determine a safe reorder point from cumulative demand paths
5. Demonstrate concretely that marginal quantiles give wrong answers for all three questions

## Prerequisites

- Notebook 01: Generating Sample Paths (provides `paths` array and `forecasts` DataFrame)
- Guide 01: Sample Paths Theory

In [ ]:
learning_objectives([
    "**Time:** ~15 minutes",
    "Notebook 01: Generating Sample Paths (provides `paths` array and `forecasts` DataFrame)",
    "Guide 01: Sample Paths Theory"
])

## Setup: Regenerate Paths

We retrain the model and regenerate paths here so this notebook is self-contained. If you have the model and paths from Notebook 01, skip to Section 1.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

np.random.seed(42)

# ----- Load data -----
url = (
    "https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series"
    "/main/datasets/french_bakery.csv"
)
df_raw = pd.read_csv(url, parse_dates=["date"])
df = (
    df_raw[df_raw["item"] == "BAGUETTE"][["date", "quantity"]]
    .copy()
    .rename(columns={"date": "ds", "quantity": "y"})
    .assign(unique_id="baguette")
    [["unique_id", "ds", "y"]]
    .sort_values("ds")
    .reset_index(drop=True)
)

H = 7
df_train = df.iloc[:-H].copy()
df_test = df.iloc[-H:].copy()

# ----- Train model -----
model = NHITS(
    h=7,
    input_size=28,
    loss=MQLoss(level=[80, 90]),
    scaler_type="robust",
    max_steps=500,
)
nf = NeuralForecast(models=[model], freq="D")
nf.fit(df_train)

# ----- Generate paths -----
N_PATHS = 500   # use 500 for stable probability estimates
paths_df = nf.models[0].simulate(n_paths=N_PATHS)
forecasts = nf.predict()

path_cols = [c for c in paths_df.columns if c.startswith("sample_")]
paths = paths_df[path_cols].values.T   # (N_PATHS, H): rows=paths, cols=horizon steps

print(f"paths shape:    {paths.shape}")
print(f"forecasts shape: {forecasts.shape}")
print("Setup complete.")

## The Universal Monte Carlo Template

Every business question in this notebook follows the same three-step pattern:

```
1. SIMULATE   → paths array of shape (N_PATHS, H)
2. APPLY      → compute f(path) for each row
3. AGGREGATE  → np.quantile(results, service_level)
```

Only the function `f` changes between questions.

In [ ]:
def monte_carlo(paths: np.ndarray, f, quantile: float = 0.80) -> float:
    """
    Apply the Monte Carlo framework to answer a business question.

    Parameters
    ----------
    paths    : np.ndarray (n_paths, horizon)
    f        : callable, applied to each path (shape: horizon,)
    quantile : service-level quantile to report

    Returns
    -------
    float — the `quantile` percentile of f applied across all paths
    """
    results = np.array([f(paths[s]) for s in range(len(paths))])
    return float(np.quantile(results, quantile))


def monte_carlo_distribution(paths: np.ndarray, f) -> np.ndarray:
    """Return the full distribution of f(path) across all paths."""
    return np.array([f(paths[s]) for s in range(len(paths))])


print("Monte Carlo template defined.")

---

## Question 1: Weekly Total — How Many Baguettes Do I Need?

**Business question:** "How many baguettes should I order for the entire week to achieve an 80% service level (i.e., avoid stock-out with 80% probability)?"

**The function:** `f(path) = sum(path)` — the weekly total demand in each scenario

**The answer:** The 80th percentile of weekly totals across all paths

**Why marginal quantiles fail:** $Q_{0.80}(\sum_{t=1}^H y_t) \neq \sum_{t=1}^H Q_{0.80}(y_t)$

In [ ]:
# Correct answer via sample paths
weekly_totals = monte_carlo_distribution(paths, f=lambda p: p.sum())
correct_weekly_80 = np.quantile(weekly_totals, 0.80)

# Naive answer via marginal quantiles: sum each day's 80th pct independently
# NHITS-hi-80 is the 80th percentile marginal forecast for each step
naive_weekly_80 = forecasts["NHITS-hi-80"].sum()

# Actual test week total (the ground truth)
actual_total = df_test["y"].sum()

print("=" * 55)
print("Q1: Baguettes needed for the week at 80% service level")
print("=" * 55)
print(f"  Sample paths (CORRECT):      {correct_weekly_80:.0f} baguettes")
print(f"  Marginal quantile sum (WRONG):{naive_weekly_80:.0f} baguettes")
print(f"  Actual test week total:       {actual_total:.0f} baguettes")
print(f"\n  Difference (over-order if wrong): {naive_weekly_80 - correct_weekly_80:+.0f} baguettes")

In [ ]:
# Visualise the distribution of weekly totals
fig, ax = plt.subplots(figsize=(9, 4))

ax.hist(weekly_totals, bins=40, color="steelblue", alpha=0.75, edgecolor="white",
        label="Weekly total per path")

ax.axvline(correct_weekly_80, color="navy", linewidth=2.5,
           label=f"80th pct from paths: {correct_weekly_80:.0f}")
ax.axvline(naive_weekly_80, color="crimson", linewidth=2.5, linestyle="--",
           label=f"Sum of marginal 80th pcts: {naive_weekly_80:.0f}")
ax.axvline(actual_total, color="forestgreen", linewidth=2, linestyle=":",
           label=f"Actual total: {actual_total:.0f}")

ax.set_xlabel("Weekly total demand (baguettes)")
ax.set_ylabel("Number of paths")
ax.set_title("Distribution of Weekly Total Demand\nSample paths vs. naive marginal sum")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("q1_weekly_total.png", dpi=150, bbox_inches="tight")
plt.show()

The histogram shows the distribution of weekly totals across all 500 paths. The navy line is the correct 80th percentile. The red dashed line is the wrong naive answer — it is shifted right, meaning a bakery using marginal quantiles would systematically over-order.

---

## Question 2: Worst-Case Single Day — What Is My Peak Demand Risk?

**Business question:** "For capacity planning (e.g., oven slots, staff rostering), what is the worst single day I should plan for at an 80% confidence level?"

**The function:** `f(path) = max(path)` — the worst day in each simulated week

**The answer:** The 80th percentile of per-path maxima

**Why marginal quantiles fail:** The 80th percentile of $\max_t y_t$ is not the maximum of the marginal 80th percentiles. The maximum introduces a joint probability across days.

In [ ]:
# Correct answer via sample paths
worst_days = monte_carlo_distribution(paths, f=lambda p: p.max())
correct_worst_80 = np.quantile(worst_days, 0.80)

# Naive answer: maximum of marginal 80th percentiles
# This gives the 80th pct for each day; the max is not the right operation here
naive_worst_80 = forecasts["NHITS-hi-80"].max()

# Also consider: what if someone just takes max of marginal medians?
naive_worst_median = forecasts["NHITS"].max()

# Actual worst day
actual_worst = df_test["y"].max()

print("=" * 55)
print("Q2: Worst-case single day at 80% confidence")
print("=" * 55)
print(f"  Sample paths 80th pct of max (CORRECT): {correct_worst_80:.0f}")
print(f"  Max of marginal 80th pcts (WRONG):       {naive_worst_80:.0f}")
print(f"  Max of marginal medians (also wrong):    {naive_worst_median:.0f}")
print(f"  Actual worst day:                        {actual_worst:.0f}")
print()
print("  The correct answer accounts for the probability that")
print("  the worst day falls on any of the 7 days — a joint event.")

In [ ]:
# Visualise worst-day distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: distribution of per-path worst day
ax = axes[0]
ax.hist(worst_days, bins=35, color="darkorange", alpha=0.75, edgecolor="white")
ax.axvline(correct_worst_80, color="navy", linewidth=2.5,
           label=f"80th pct from paths: {correct_worst_80:.0f}")
ax.axvline(naive_worst_80, color="crimson", linewidth=2, linestyle="--",
           label=f"Max of marginal 80th pcts: {naive_worst_80:.0f}")
ax.axvline(actual_worst, color="forestgreen", linewidth=2, linestyle=":",
           label=f"Actual worst: {actual_worst:.0f}")
ax.set_xlabel("Worst single day (baguettes)")
ax.set_ylabel("Number of paths")
ax.set_title("Distribution of Worst Single Day\nacross 500 paths")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Right: compare distributions of daily demand on each horizon day
ax = axes[1]
for t in range(H):
    ax.boxplot(paths[:, t], positions=[t + 1], widths=0.6,
               patch_artist=True,
               boxprops=dict(facecolor="steelblue", alpha=0.5),
               medianprops=dict(color="navy", linewidth=2),
               flierprops=dict(marker=".", markersize=2, alpha=0.4))
ax.set_xticks(range(1, H + 1))
ax.set_xticklabels([f"Day {t}" for t in range(1, H + 1)])
ax.set_xlabel("Forecast day")
ax.set_ylabel("Demand")
ax.set_title("Distribution of Demand per Day\nacross 500 paths")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("q2_worst_day.png", dpi=150, bbox_inches="tight")
plt.show()

The left panel shows the distribution of per-path worst days. The 80th percentile of this distribution is the correct capacity planning number. The right panel shows that any individual day has lower expected demand than the weekly worst — the maximum over the week is a compound event.

---

## Question 3: Reorder Timing — When Should I Place My Next Order?

**Business question:** "I have `STOCK` baguettes on hand at the start of the week. By which day should I place a reorder to ensure I don't run out with 80% probability?"

**The approach:**
1. For each path, compute the cumulative sum of demand
2. Find the first day the cumulative sum exceeds `STOCK` (the stock-out day)
3. The 20th percentile of stock-out days = the latest safe reorder point (80% of paths run out after this day)

**Why marginal quantiles fail:** Cumulative demand across days is a path-dependent quantity. Each day's marginal distribution tells you nothing about when the running total crosses a threshold.

In [ ]:
# Current stock level — set to the median weekly forecast
STOCK = float(np.median(paths.sum(axis=1))) * 0.75   # 75% of median weekly demand
STOCK = round(STOCK / 10) * 10   # round to nearest 10 for a clean example

print(f"Assumed starting stock: {STOCK:.0f} baguettes")
print(f"(75% of median weekly forecast: {np.median(paths.sum(axis=1)):.0f})")

In [ ]:
def stockout_day(path: np.ndarray, stock: float) -> float:
    """
    Find the first day where cumulative demand exceeds `stock`.

    Parameters
    ----------
    path  : 1D array of shape (H,) — one simulated demand trajectory
    stock : float — starting inventory level

    Returns
    -------
    int — day number (1-indexed) of stock-out; H+1 if no stock-out this week
    """
    cumulative = np.cumsum(path)
    crossing = np.where(cumulative > stock)[0]
    if len(crossing) == 0:
        return H + 1   # no stock-out this week
    return int(crossing[0]) + 1   # 1-indexed day number


# Apply across all paths
stockout_days = np.array([stockout_day(paths[s], STOCK) for s in range(N_PATHS)])

print("Stock-out day distribution:")
for day in range(1, H + 2):
    label = f"Day {day}" if day <= H else "No stock-out"
    count = (stockout_days == day).sum()
    bar = "█" * (count // 3)
    print(f"  {label:12s}: {count:3d} paths  {bar}")

In [ ]:
# Safe reorder point: the 20th percentile of stock-out days
# "80% of paths run out after this day" → reorder by this day to be safe
reorder_day = int(np.quantile(stockout_days, 0.20))

# What fraction of paths experience stock-out during the week?
stockout_prob = (stockout_days <= H).mean()

print("=" * 55)
print("Q3: Safe reorder timing")
print("=" * 55)
print(f"  Starting stock:        {STOCK:.0f} baguettes")
print(f"  P(stock-out this week):{stockout_prob:.1%}")
print(f"  Safe reorder by day:   {reorder_day}  (80% service level)")
print()
print("  Marginal quantiles cannot answer this question at all:")
print("  there is no per-step marginal quantity that tells you")
print("  the distribution of CUMULATIVE demand crossing a threshold.")

In [ ]:
# Visualise cumulative demand paths with the stock threshold
cumulative_paths = np.cumsum(paths, axis=1)   # (N_PATHS, H)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: cumulative demand paths vs. stock level
ax = axes[0]
for s in range(min(N_PATHS, 200)):   # plot first 200 for clarity
    color = "crimson" if stockout_days[s] <= H else "steelblue"
    ax.plot(range(1, H + 1), cumulative_paths[s], color=color, alpha=0.07, linewidth=0.9)

ax.axhline(STOCK, color="black", linewidth=2, linestyle="--",
           label=f"Stock level: {STOCK:.0f}")
ax.axvline(reorder_day, color="navy", linewidth=2,
           label=f"Reorder by day {reorder_day} (80% safe)")

# Median cumulative demand
ax.plot(range(1, H + 1), np.median(cumulative_paths, axis=0),
        color="darkblue", linewidth=2.5, label="Median cumulative")

red_patch = mpatches.Patch(color="crimson", alpha=0.5, label="Stock-out paths")
blue_patch = mpatches.Patch(color="steelblue", alpha=0.5, label="No stock-out paths")
ax.legend(handles=[red_patch, blue_patch,
                   plt.Line2D([0], [0], color="black", linestyle="--", label=f"Stock: {STOCK:.0f}"),
                   plt.Line2D([0], [0], color="navy", label=f"Reorder by day {reorder_day}")],
          fontsize=9)

ax.set_xlabel("Forecast day")
ax.set_ylabel("Cumulative demand")
ax.set_title("Cumulative Demand Paths\nRed = stock-out, Blue = sufficient stock")
ax.grid(alpha=0.3)

# Right: distribution of stock-out days
ax = axes[1]
bins = np.arange(0.5, H + 2.5)
ax.hist(stockout_days, bins=bins, color="coral", edgecolor="white",
        alpha=0.8, label="Stock-out day")
ax.axvline(reorder_day, color="navy", linewidth=2.5,
           label=f"20th pct (reorder by day {reorder_day})")
ax.set_xticks(range(1, H + 2))
ax.set_xticklabels([f"Day {d}" if d <= H else "None" for d in range(1, H + 2)],
                   rotation=30)
ax.set_xlabel("Day of stock-out")
ax.set_ylabel("Number of paths")
ax.set_title("When Does Stock Run Out?\nDistribution across 500 paths")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("q3_reorder_timing.png", dpi=150, bbox_inches="tight")
plt.show()

The left panel shows cumulative demand across paths, colored by whether they exceed the stock level. The right panel shows the distribution of stock-out days — the 20th percentile gives the latest day by which a reorder achieves an 80% service level.

---

## Summary: Why Marginal Quantiles Fail for All Three Questions

In [ ]:
print("=" * 60)
print("THREE QUESTIONS — TWO METHODS — ONE CORRECT")
print("=" * 60)
print()
print("Q1: Weekly total at 80% service level")
print(f"   Correct (sample paths):      {correct_weekly_80:.0f} baguettes")
print(f"   Wrong (marginal sum):         {naive_weekly_80:.0f} baguettes")
print(f"   Actual test week total:       {actual_total:.0f} baguettes")
print()
print("Q2: Worst-case single day at 80% confidence")
print(f"   Correct (sample paths):      {correct_worst_80:.0f} baguettes")
print(f"   Wrong (max of marginal 80%): {naive_worst_80:.0f} baguettes")
print(f"   Actual worst day:             {actual_worst:.0f} baguettes")
print()
print("Q3: Reorder timing at 80% service level")
print(f"   Correct (sample paths):      Reorder by day {reorder_day}")
print(f"   Wrong (marginal quantiles):  Cannot answer — no single")
print(f"                                 marginal captures cumulative crossing")
print()
print("Key insight: all three questions involve functions of the")
print("JOINT distribution across multiple steps. Marginal quantiles")
print("treat each step independently and cannot answer them correctly.")

## Extension: Your Own Business Question

The Monte Carlo framework is general. Define any function `f(path)` and call `monte_carlo(paths, f, quantile)`. Some ideas to try:

- **Order sizing for sub-intervals:** What is the 90th percentile of demand in days 3-5 only?
- **Consecutive high-demand days:** How likely is a run of 3+ days above 150 units?
- **Revenue at risk:** If you can only produce 120 units/day, what fraction of days is demand unmet?

In [ ]:
# Example: 90th percentile of demand in days 3-5 only
midweek_90 = monte_carlo(
    paths,
    f=lambda p: p[2:5].sum(),   # days 3, 4, 5 (0-indexed: 2, 3, 4)
    quantile=0.90
)
print(f"90th pct of days 3-5 total: {midweek_90:.0f} baguettes")

# Example: probability of 3+ consecutive days above a threshold
THRESHOLD = float(np.quantile(paths, 0.70))  # 70th pct of all daily values

def has_three_consecutive_high(path, threshold=THRESHOLD):
    """Return 1 if any 3 consecutive days exceed threshold."""
    for start in range(len(path) - 2):
        if all(path[start:start+3] > threshold):
            return 1
    return 0

prob_run = monte_carlo_distribution(paths, has_three_consecutive_high).mean()
print(f"P(3+ consecutive days > {THRESHOLD:.0f}): {prob_run:.2%}")

# Example: capacity at risk
CAPACITY = float(np.median(paths.max(axis=1))) * 0.85  # 85% of median peak
unmet_days = monte_carlo_distribution(
    paths,
    f=lambda p: (p > CAPACITY).sum()
)
print(f"\nExpected days with unmet demand (capacity={CAPACITY:.0f}): {unmet_days.mean():.2f} days/week")

In [ ]:
section_divider("Summary")

## Summary

| Business question | Function $f$ | Answer | Marginals work? |
|---|---|---|---|
| Weekly total at 80% SL | `p.sum()` | 80th pct of sums | No |
| Worst-case day at 80% | `p.max()` | 80th pct of maxima | No |
| Reorder by day $d$ | Cumulative threshold crossing | 20th pct of crossing days | No |
| Mid-week sub-interval | `p[2:5].sum()` | Any quantile of sums | No |
| Probability of any event | `indicator(event)` | Mean across paths | No |

Marginal quantiles only answer single-step, single-quantile questions. Every multi-step or joint question requires sample paths.

## Next: Exercises

The self-check exercises in `exercises/01_sample_path_exercises.py` reinforce path shape, weekly totals, and reorder timing with assertion-based tests.

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])